In [36]:
# ==============================================================================
# Этап 1: Инициализация пайплайна и фильтрация целевой переменной
# Проект: FutureScore (Decentrathon 5.0)
# ==============================================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import pickle

# 1. Загрузка сырых данных (предобработанных Участником А)
try:
    df = pd.read_csv('features.csv')
    print(f"📦 Исходный датасет загружен. Размер: {df.shape}")
except FileNotFoundError:
    print("❌ Ошибка: Файл 'features.csv' не найден. Загрузите его в среду.")

# 2. Очистка "серой зоны" (оставляем только финальные статусы для честного обучения)
clear_statuses = ['Исполнена', 'Одобрена', 'Отклонена', 'Отозвано']
df_clean = df[df['Статус заявки'].isin(clear_statuses)].copy()

# 3. Формирование Target (1 - Успех, 0 - Провал/Отказ)
success_statuses = ['Исполнена', 'Одобрена']
df_clean['target'] = df_clean['Статус заявки'].isin(success_statuses).astype(int)

print(f"🧹 Очистка завершена. Рабочий объем: {len(df_clean)} строк.")
print(f"🎯 Баланс классов:\n{df_clean['target'].value_counts(normalize=True).round(3) * 100}")

📦 Исходный датасет загружен. Размер: (36651, 11)
🧹 Очистка завершена. Рабочий объем: 33600 строк.
🎯 Баланс классов:
target
1    85.2
0    14.8
Name: proportion, dtype: float64


In [37]:
# ==============================================================================
# Этап 2: Обогащение данных (Data Enrichment)
# Интеграция климатического индекса на базе открытых данных (eGov, Казгидромет)
# ==============================================================================

# Справочник рисков (1.0 - макс. риск засухи/неурожая, 0.0 - идеальные условия)
climate_risk_map = {
    'Кызылординская область': 0.90, 'Мангистауская область': 0.95,
    'Атырауская область': 0.85, 'Туркестанская область': 0.70,
    'г.Шымкент': 0.60, 'Жамбылская область': 0.75,
    'область Ұлытау': 0.80, 'Карагандинская область': 0.60,
    'Актюбинская область': 0.65, 'область Абай': 0.55,
    'Западно-Казахстанская область': 0.50, 'Алматинская область': 0.40,
    'область Жетісу': 0.45, 'Восточно-Казахстанская область': 0.35,
    'Павлодарская область': 0.40, 'Костанайская область': 0.30,
    'Акмолинская область': 0.25, 'Северо-Казахстанская область': 0.20
}

# Безопасный маппинг (если региона нет в справочнике, ставим средний риск 0.5)
df_clean['climate_risk'] = df_clean['region'].map(climate_risk_map).fillna(0.5)

print("🌍 Внешние климатические данные успешно интегрированы.")

🌍 Внешние климатические данные успешно интегрированы.


In [40]:
# ==============================================================================
# Этап 3: Генерация признаков (Feature Engineering)
# Оцифровка бизнес-логики Правил субсидирования РК
# ==============================================================================

# 1. Временные признаки (Сезонность)
df_clean['date'] = pd.to_datetime(df_clean['date'], errors='coerce')
df_clean['month'] = df_clean['date'].dt.month.fillna(6).astype(int) # Если даты нет, ставим лето

# 2. Финансовые метрики (Адекватность запроса)
# Защита от деления на ноль (+1)
df_clean['amount_to_norm_ratio'] = df_clean['Причитающая сумма'] / (df_clean['Норматив'] + 1)

# 3. Индикаторы стратегического развития (Merit-based инновации)
df_clean['is_breeding'] = df_clean['Наименование субсидирования'].str.contains('племен', case=False, na=False).astype(int)
df_clean['is_selection'] = df_clean['Наименование субсидирования'].str.contains('селекцион', case=False, na=False).astype(int)

# 4. Географический потенциал (Исторический Success Rate района)
# Считаем средний процент успеха по району
district_score_map = df_clean.groupby('Район хозяйства')['target'].mean().to_dict()
global_mean_score = df_clean['target'].mean()

# Применяем справочник к датасету
df_clean['district_historical_score'] = df_clean['Район хозяйства'].map(district_score_map).fillna(global_mean_score)

print("🧠 Feature Engineering завершен. Созданы предиктивные признаки.")

🧠 Feature Engineering завершен. Созданы предиктивные признаки.


In [41]:
# ==============================================================================
# Этап 4: Label Encoding и экспорт артефактов пайплайна
# Подготовка матриц для алгоритма машинного обучения и веб-дашборда
# ==============================================================================

categorical_cols = ['region', 'Район хозяйства', 'Направление водства', 'Акимат']
label_encoders = {}

# Заполняем пустоты перед кодированием
df_clean[categorical_cols] = df_clean[categorical_cols].fillna('Неизвестно')

# Переводим текст в числа
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col + '_encoded'] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le

# Финальный список признаков, которые "увидит" ИИ
features = [
    'month', 'amount_to_norm_ratio', 'is_breeding', 'is_selection',
    'district_historical_score', 'climate_risk',
    'region_encoded', 'Район хозяйства_encoded', 'Направление водства_encoded'
]

# Упаковываем "память" обработки для передачи на сайт (Участнику В/С)
pipeline_artifacts = {
    'label_encoders': label_encoders,
    'district_score_map': district_score_map,
    'climate_risk_map': climate_risk_map,
    'global_mean_score': global_mean_score,
    'features_list': features
}

# Сохраняем чистый датасет для обучения
df_clean[features + ['target']].to_csv('final_dataset_pro.csv', index=False)

# Сохраняем артефакты
with open('data_pipeline_artifacts_pro.pkl', 'wb') as f:
    pickle.dump(pipeline_artifacts, f)

print(f"✅ УСПЕШНО! Экспортировано {len(features)} признаков.")
print("Файлы 'final_dataset_pro.csv' и 'data_pipeline_artifacts_pro.pkl' готовы к загрузке в Model_Training.")

✅ УСПЕШНО! Экспортировано 9 признаков.
Файлы 'final_dataset_pro.csv' и 'data_pipeline_artifacts_pro.pkl' готовы к загрузке в Model_Training.
